# barrier_action_label_purged_bc_v1

Notebook-owned diagnostic for the active objective: improve excess toward `+30pp`, keep key-window MaxDD below `15%`, and avoid overfitting with purged expanding-window validation.

Governance:
- Canonical baseline: `runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv`.
- Direct comparison baselines: `two_stage_reentry_derisk_purged_bc_v1` and `tree_reentry_derisk_purged_bc_v1` because both found that old forward-return/drawdown labels were not enough.
- Role: `diagnose`; no actor checkpoint and no runtime promotion from this notebook alone.
- Changed layer: `model` / `model_selection` through label quality, not reward or execution thresholds.
- Reusable code: `src/soxl_quant/app/barrier_labels.py` computes close-to-close triple-barrier outcomes and staged action labels; experiment-specific feature sets, grids, thresholds, and conclusions stay in this notebook.
- Target windows: `wf_2022`, `wf_2023_2024`, shared bad-window clusters `C002_2025_rebound` and `C003_2026_reentry`.
- Gate: mean relative-return delta `>= +30pp`, every key-window MaxDD `<= 15%`, and no-worse-window guard versus baseline.
- External references: Qlib (`https://github.com/microsoft/qlib`, `https://arxiv.org/abs/2009.11189`) for non-leaking experiment workflow; Lopez de Prado triple-barrier/meta-labeling and purged CV concepts; behavior-cloning trading-agent reference (`https://arxiv.org/abs/2110.01325`).

Closest historical success case:
- `predictive_drawdown_risk_gate_v1` found strict OOS candidates with `relative_return > +30pp` and MaxDD near/under `15%`, but failed saved walk-forward gates.

Closest historical failure case:
- `tree_reentry_derisk_purged_bc_v1` showed that changing model class alone produced only `+1.32pp` mean relative delta and `31.10%` MaxDD.

Why this is not a duplicate:
- It changes the label definition to first-hit barrier outcomes: whether profit-taking or stop-loss happens first before the vertical horizon. This tests label quality directly instead of another model-class or threshold-only sweep.


In [1]:
from __future__ import annotations

import json
from dataclasses import asdict
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.soxl_quant.app.barrier_labels import BarrierActionLabelConfig, TripleBarrierConfig, build_barrier_action_label_frame
from src.soxl_quant.app.budget_counterfactual import compute_metrics
from src.soxl_quant.app.oos_diagnostics import load_qlib_feature_frame
from src.soxl_quant.app.purged_walkforward import build_purged_walkforward_splits, single_position_frame

EXPERIMENT = "barrier_action_label_purged_bc_v1"
OUTPUT_DIR = Path("runtime/model/card") / EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_DAILY_CSV = Path("runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv")
LOGISTIC_SELECTION_JSON = Path("runtime/model/card/two_stage_reentry_derisk_purged_bc_v1/selection.json")
TREE_SELECTION_JSON = Path("runtime/model/card/tree_reentry_derisk_purged_bc_v1/selection.json")
DAILY = pd.read_csv(BASELINE_DAILY_CSV, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
PRICE = DAILY.set_index("datetime")["price"].astype(float)
FEATURES = load_qlib_feature_frame(str(DAILY["datetime"].min().date()), str(DAILY["datetime"].max().date()))
FEATURES = FEATURES.reset_index().rename(columns={"index": "datetime"})
FEATURES["datetime"] = pd.to_datetime(FEATURES["datetime"])

WINDOWS = {
    "wf_2022": ("2022-01-01", "2022-12-31"),
    "wf_2023_2024": ("2023-01-01", "2024-12-31"),
    "bad_C002_2025_rebound": ("2025-04-04", "2025-07-22"),
    "bad_C003_2026_reentry": ("2026-02-04", "2026-05-21"),
}
FEATURE_SETS = {
    "cluster_vol_reentry": [
        "vol_of_vol_21", "tracking_error_vol_21", "decay_63", "realized_vol_63", "tracking_error_21",
        "realized_vol_21", "mom_21", "mom_10", "relative_strength_21", "vix_percentile_252",
        "anchor_gap_180", "below_anchor_ratio_63", "regime_risk", "drawdown_63", "downside_semivol_21",
        "vix_zscore", "trend_gap_21", "trend_gap_63", "realized_vol_10", "days_since_high_63",
    ],
    "cluster_vol_compact": [
        "vol_of_vol_21", "tracking_error_vol_21", "decay_63", "realized_vol_63", "tracking_error_21",
        "realized_vol_21", "vix_percentile_252", "regime_risk", "drawdown_63", "days_since_high_63",
    ],
}
for name, cols in FEATURE_SETS.items():
    missing = [col for col in cols if col not in FEATURES.columns]
    if missing:
        raise ValueError(f"feature set {name} missing columns: {missing}")

logistic_selection = json.loads(LOGISTIC_SELECTION_JSON.read_text()) if LOGISTIC_SELECTION_JSON.exists() else {}
tree_selection = json.loads(TREE_SELECTION_JSON.read_text()) if TREE_SELECTION_JSON.exists() else {}
print({
    "daily_rows": len(DAILY),
    "daily_start": str(DAILY["datetime"].min().date()),
    "daily_end": str(DAILY["datetime"].max().date()),
    "feature_rows": len(FEATURES),
    "feature_sets": {name: len(cols) for name, cols in FEATURE_SETS.items()},
    "logistic_decision": logistic_selection.get("decision"),
    "tree_decision": tree_selection.get("decision"),
})

[74434:MainThread](2026-05-25 13:30:27,750) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[74434:MainThread](2026-05-25 13:30:28,471) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[74434:MainThread](2026-05-25 13:30:28,471) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[74434:MainThread](2026-05-25 13:30:31,032) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[74434:MainThread](2026-05-25 13:30:31,033) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[74434:MainThread](2026-05-25 13:30:31,040) INFO - qlib.timer - [log.py:127] - Time cost: 0.009s | Init data Done


{'daily_rows': 1605, 'daily_start': '2020-01-02', 'daily_end': '2026-05-21', 'feature_rows': 1605, 'feature_sets': {'cluster_vol_reentry': 20, 'cluster_vol_compact': 10}, 'logistic_decision': 'diagnostic_not_promoted', 'tree_decision': 'diagnostic_not_promoted'}


In [2]:
def score_positions(frame: pd.DataFrame, start: str, end: str, position_col: str = "position") -> dict:
    sub = frame[(frame["datetime"] >= start) & (frame["datetime"] <= end)].copy()
    return compute_metrics(single_position_frame(sub, position_col=position_col), PRICE.loc[start:end])


def base_window_table() -> pd.DataFrame:
    rows = []
    for name, (start, end) in WINDOWS.items():
        rows.append({"window": name, "variant": "baseline", **score_positions(DAILY, start, end)})
    return pd.DataFrame(rows)


BASE_WINDOWS = base_window_table()
BASE_WINDOWS[["window", "return", "relative_return", "max_dd", "avg_pos", "trades"]]

,window,return,relative_return,max_dd,avg_pos,trades
0,wf_2022,-21.567182,65.020890,31.096849,0.248277,26
1,wf_2023_2024,209.141657,17.368148,30.668184,0.497231,70
2,bad_C002_2025_rebound,76.882481,-124.606649,16.324924,0.572851,9
3,bad_C003_2026_reentry,17.799626,-217.582972,6.421473,0.161722,2


In [3]:
def build_labeled_feature_frame(cfg: BarrierActionLabelConfig, feature_cols: list[str]) -> pd.DataFrame:
    labels = build_barrier_action_label_frame(DAILY, cfg)
    # Stage actions into independent binary labels for auditable BC diagnostics.
    labels["staged_reentry_label"] = labels["reentry_label"] & (~labels["derisk_label"])
    frame = labels.merge(FEATURES[["datetime", *feature_cols]], on="datetime", how="left")
    return frame.dropna(subset=feature_cols).reset_index(drop=True)


def binary_probability(train: pd.DataFrame, test: pd.DataFrame, label_col: str, *, feature_cols: list[str], c_value: float) -> np.ndarray:
    y = train[label_col].astype(int)
    positives = int(y.sum())
    negatives = int((1 - y).sum())
    if positives < 5 or negatives < 5:
        return np.zeros(len(test), dtype=float)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=float(c_value), class_weight="balanced", max_iter=1000, random_state=17),
    )
    model.fit(train[feature_cols], y)
    classes = list(model.named_steps["logisticregression"].classes_)
    if 1 not in classes:
        return np.zeros(len(test), dtype=float)
    return model.predict_proba(test[feature_cols])[:, classes.index(1)]


def apply_barrier_action_path(
    test: pd.DataFrame,
    *,
    cfg: BarrierActionLabelConfig,
    p_derisk: np.ndarray,
    p_reentry: np.ndarray,
    derisk_threshold: float,
    clear_threshold: float,
    reentry_threshold: float,
) -> pd.DataFrame:
    out = test[[
        "datetime", "base_position", "barrier_label", "derisk_label", "staged_reentry_label",
        "forward_return", "max_forward_return", "max_forward_drawdown",
    ]].copy()
    out = out.rename(columns={"base_position": "position"})
    base = out["position"].to_numpy(dtype=float)
    derisk_mask = (p_derisk >= float(derisk_threshold)) & (base >= float(cfg.min_derisk_position))
    reentry_mask = (
        (p_derisk <= float(clear_threshold))
        & (p_reentry >= float(reentry_threshold))
        & (base <= float(cfg.max_reentry_position))
        & (~derisk_mask)
    )
    out["base_position"] = base
    out.loc[derisk_mask, "position"] = np.minimum(out.loc[derisk_mask, "position"], float(cfg.derisk_position))
    out.loc[reentry_mask, "position"] = np.maximum(out.loc[reentry_mask, "position"], float(cfg.reentry_position))
    out["p_derisk"] = p_derisk
    out["p_reentry"] = p_reentry
    out["derisk_pred"] = derisk_mask
    out["reentry_pred"] = reentry_mask
    out["position_delta"] = out["position"] - out["base_position"]
    return out

In [4]:
def score_candidate(
    *,
    cfg: BarrierActionLabelConfig,
    feature_set_name: str,
    feature_cols: list[str],
    c_value: float,
    derisk_threshold: float,
    clear_threshold: float,
    reentry_threshold: float,
) -> tuple[dict, list[dict], dict[str, pd.DataFrame], list[dict]]:
    frame = build_labeled_feature_frame(cfg, feature_cols)
    splits = build_purged_walkforward_splits(
        frame,
        WINDOWS,
        purge_rows=int(cfg.barrier.horizon_days),
        min_train_rows=120,
        min_test_rows=2,
    )
    row = {
        "barrier_horizon_days": int(cfg.barrier.horizon_days),
        "barrier_upper_return": float(cfg.barrier.upper_return),
        "barrier_lower_return": float(cfg.barrier.lower_return),
        "max_reentry_position": float(cfg.max_reentry_position),
        "min_derisk_position": float(cfg.min_derisk_position),
        "reentry_position": float(cfg.reentry_position),
        "derisk_position": float(cfg.derisk_position),
        "feature_set": feature_set_name,
        "feature_count": len(feature_cols),
        "c_value": float(c_value),
        "derisk_threshold": float(derisk_threshold),
        "clear_threshold": float(clear_threshold),
        "reentry_threshold": float(reentry_threshold),
        "split_count": len(splits),
        "total_derisk_predictions": 0,
        "total_reentry_predictions": 0,
        "total_derisk_true": 0,
        "total_reentry_true": 0,
    }
    comparison_rows = []
    audit_rows = []
    path_by_window: dict[str, pd.DataFrame] = {}
    deltas = []
    max_dds = []
    no_worse_windows = 0
    for split in splits:
        train = split.train[split.train["has_forward_window"]].copy()
        test = split.test.copy()
        p_derisk = binary_probability(train, test, "derisk_label", feature_cols=feature_cols, c_value=c_value)
        p_reentry = binary_probability(train, test, "staged_reentry_label", feature_cols=feature_cols, c_value=c_value)
        path = apply_barrier_action_path(
            test,
            cfg=cfg,
            p_derisk=p_derisk,
            p_reentry=p_reentry,
            derisk_threshold=derisk_threshold,
            clear_threshold=clear_threshold,
            reentry_threshold=reentry_threshold,
        )
        path_by_window[split.name] = path
        start, end = WINDOWS[split.name]
        metrics = score_positions(path, start, end)
        base = BASE_WINDOWS[BASE_WINDOWS["window"] == split.name].iloc[0].to_dict()
        delta_return = metrics["return"] - float(base["return"])
        delta_relative = metrics["relative_return"] - float(base["relative_return"])
        deltas.append(delta_relative)
        max_dds.append(metrics["max_dd"])
        if delta_relative >= -1e-9 and metrics["max_dd"] <= float(base["max_dd"]) + 1e-9:
            no_worse_windows += 1
        comparison_rows.append({"window": split.name, "variant": "baseline", **base})
        comparison_rows.append({"window": split.name, "variant": "barrier_bc", **metrics, "delta_return": delta_return, "delta_relative_return": delta_relative})
        derisk_pred = path["derisk_pred"].astype(bool)
        reentry_pred = path["reentry_pred"].astype(bool)
        derisk_true = path["derisk_label"].astype(bool)
        reentry_true = path["staged_reentry_label"].astype(bool)
        audit_rows.append({
            "window": split.name,
            "train_rows": int(len(train)),
            "test_rows": int(len(test)),
            "barrier_positive": int((test["barrier_label"] == 1).sum()),
            "barrier_negative": int((test["barrier_label"] == -1).sum()),
            "derisk_pred": int(derisk_pred.sum()),
            "derisk_true": int(derisk_true.sum()),
            "derisk_tp": int((derisk_pred & derisk_true).sum()),
            "derisk_fp": int((derisk_pred & ~derisk_true).sum()),
            "reentry_pred": int(reentry_pred.sum()),
            "reentry_true": int(reentry_true.sum()),
            "reentry_tp": int((reentry_pred & reentry_true).sum()),
            "reentry_fp": int((reentry_pred & ~reentry_true).sum()),
        })
        row["total_derisk_predictions"] += int(derisk_pred.sum())
        row["total_reentry_predictions"] += int(reentry_pred.sum())
        row["total_derisk_true"] += int(derisk_true.sum())
        row["total_reentry_true"] += int(reentry_true.sum())
    row["mean_delta_relative_return"] = float(np.mean(deltas)) if deltas else float("nan")
    row["min_delta_relative_return"] = float(np.min(deltas)) if deltas else float("nan")
    row["max_key_window_dd"] = float(np.max(max_dds)) if max_dds else float("nan")
    row["return_gate"] = bool(deltas and row["mean_delta_relative_return"] >= 30.0)
    row["maxdd_gate"] = bool(max_dds and row["max_key_window_dd"] <= 15.0)
    row["no_worse_window_count"] = int(no_worse_windows)
    row["no_worse_gate"] = bool(no_worse_windows == len(splits) and len(splits) == len(WINDOWS))
    row["full_gate"] = bool(row["return_gate"] and row["maxdd_gate"] and row["no_worse_gate"])
    fp_penalty = sum(a["derisk_fp"] + a["reentry_fp"] for a in audit_rows) * 0.05
    action_penalty = 15.0 if row["total_derisk_predictions"] + row["total_reentry_predictions"] == 0 else 0.0
    row["diagnostic_score"] = float(row["mean_delta_relative_return"] - 1.5 * max(0.0, row["max_key_window_dd"] - 15.0) - fp_penalty - action_penalty)
    return row, comparison_rows, path_by_window, audit_rows

In [5]:
BARRIER_CONFIGS = [
    BarrierActionLabelConfig(TripleBarrierConfig(21, 0.20, 0.08), max_reentry_position=0.35, min_derisk_position=0.50, reentry_position=0.75, derisk_position=0.0),
    BarrierActionLabelConfig(TripleBarrierConfig(21, 0.25, 0.10), max_reentry_position=0.55, min_derisk_position=0.55, reentry_position=0.75, derisk_position=0.0),
    BarrierActionLabelConfig(TripleBarrierConfig(42, 0.30, 0.12), max_reentry_position=0.55, min_derisk_position=0.55, reentry_position=0.75, derisk_position=0.0),
]
THRESHOLD_TRIPLES = [
    (0.55, 0.50, 0.55),
    (0.65, 0.50, 0.65),
    (0.75, 0.55, 0.75),
]

rows = []
comparison_by_key = {}
path_by_key = {}
audit_by_key = {}
for cfg, (feature_set_name, feature_cols), c_value, (derisk_threshold, clear_threshold, reentry_threshold) in product(
    BARRIER_CONFIGS,
    FEATURE_SETS.items(),
    [0.05, 0.20],
    THRESHOLD_TRIPLES,
):
    row, comparison_rows, paths, audit_rows = score_candidate(
        cfg=cfg,
        feature_set_name=feature_set_name,
        feature_cols=feature_cols,
        c_value=c_value,
        derisk_threshold=derisk_threshold,
        clear_threshold=clear_threshold,
        reentry_threshold=reentry_threshold,
    )
    key = len(rows)
    row["candidate_key"] = key
    rows.append(row)
    comparison_by_key[key] = comparison_rows
    path_by_key[key] = paths
    audit_by_key[key] = audit_rows

results = pd.DataFrame(rows).sort_values(["full_gate", "diagnostic_score", "mean_delta_relative_return"], ascending=[False, False, False]).reset_index(drop=True)
results.to_csv(OUTPUT_DIR / "barrier_grid_results.csv", index=False)
print("grid rows", len(results), "full gate passes", int(results["full_gate"].sum()))
results.head(15)[[
    "candidate_key", "barrier_horizon_days", "barrier_upper_return", "barrier_lower_return", "feature_set", "c_value",
    "max_reentry_position", "min_derisk_position", "derisk_threshold", "clear_threshold", "reentry_threshold",
    "mean_delta_relative_return", "min_delta_relative_return", "max_key_window_dd", "return_gate", "maxdd_gate", "no_worse_gate", "full_gate",
    "total_derisk_predictions", "total_reentry_predictions", "diagnostic_score",
]]

grid rows 36 full gate passes 0


,candidate_key,barrier_horizon_days,barrier_upper_return,barrier_lower_return,feature_set,c_value,max_reentry_position,min_derisk_position,derisk_threshold,clear_threshold,...,mean_delta_relative_return,min_delta_relative_return,max_key_window_dd,return_gate,maxdd_gate,no_worse_gate,full_gate,total_derisk_predictions,total_reentry_predictions,diagnostic_score
0,14,21,0.25,0.10,cluster_vol_reentry,0.05,0.55,0.55,0.75,0.55,...,32.019269,0.000000,36.576424,True,False,False,False,52,59,-2.995368
1,32,42,0.30,0.12,cluster_vol_compact,0.05,0.55,0.55,0.75,0.55,...,13.871941,0.000000,31.096849,False,False,True,False,84,0,-12.023332
2,19,21,0.25,0.10,cluster_vol_compact,0.05,0.55,0.55,0.65,0.50,...,10.925031,0.000000,31.096849,False,False,False,False,134,10,-17.170243
3,2,21,0.20,0.08,cluster_vol_reentry,0.05,0.35,0.50,0.75,0.55,...,9.898027,-2.436878,33.630928,False,False,False,False,123,20,-20.948366
4,1,21,0.20,0.08,cluster_vol_reentry,0.05,0.35,0.50,0.65,0.50,...,19.304071,-5.432685,38.462064,False,False,False,False,188,59,-21.639025
5,23,21,0.25,0.10,cluster_vol_compact,0.20,0.55,0.55,0.75,0.55,...,4.933918,0.000000,31.096849,False,False,True,False,99,1,-21.811356
6,5,21,0.20,0.08,cluster_vol_reentry,0.20,0.35,0.50,0.75,0.55,...,15.726606,0.000000,37.681424,False,False,False,False,151,52,-22.745530
7,35,42,0.30,0.12,cluster_vol_compact,0.20,0.55,0.55,0.75,0.55,...,2.479954,0.000000,31.096849,False,False,True,False,90,0,-23.615319
8,8,21,0.20,0.08,cluster_vol_compact,0.05,0.35,0.50,0.75,0.55,...,2.718431,-8.141494,31.096849,False,False,False,False,147,3,-24.676843
9,12,21,0.25,0.10,cluster_vol_reentry,0.05,0.55,0.55,0.55,0.50,...,43.114883,-12.358725,55.115932,True,False,False,False,138,174,-26.059016


In [6]:
selected = results.iloc[0].to_dict()
selected_key = int(selected["candidate_key"])
selected_comparison = pd.DataFrame(comparison_by_key[selected_key])
selected_audit = pd.DataFrame(audit_by_key[selected_key])
selected_comparison.to_csv(OUTPUT_DIR / "selected_comparison.csv", index=False)
selected_audit.to_csv(OUTPUT_DIR / "selected_signal_audit.csv", index=False)

selected_paths = []
for window, frame in path_by_key[selected_key].items():
    out = frame.copy()
    out["window"] = window
    selected_paths.append(out)
if selected_paths:
    pd.concat(selected_paths, ignore_index=True).to_csv(OUTPUT_DIR / "selected_paths.csv", index=False)

selection = {
    "experiment": EXPERIMENT,
    "decision": "candidate_requires_saved_walkforward" if bool(selected["full_gate"]) else "diagnostic_not_promoted",
    "reason": (
        "barrier action label candidate clears notebook gates but still needs saved-actor walk-forward validation"
        if bool(selected["full_gate"])
        else "barrier action label candidates still fail the combined +30pp mean relative return, <=15% key-window MaxDD, and no-worse-window gates"
    ),
    "baseline_daily_csv": str(BASELINE_DAILY_CSV),
    "direct_comparisons": [str(LOGISTIC_SELECTION_JSON), str(TREE_SELECTION_JSON)],
    "grid_results_csv": str(OUTPUT_DIR / "barrier_grid_results.csv"),
    "selected_comparison_csv": str(OUTPUT_DIR / "selected_comparison.csv"),
    "selected_signal_audit_csv": str(OUTPUT_DIR / "selected_signal_audit.csv"),
    "selected_paths_csv": str(OUTPUT_DIR / "selected_paths.csv"),
    "selected_candidate": {k: (None if pd.isna(v) else v) for k, v in selected.items() if isinstance(v, (int, float, str, bool)) or pd.isna(v)},
    "logistic_reference": logistic_selection.get("selected_candidate", {}),
    "tree_reference": tree_selection.get("selected_candidate", {}),
    "next_path": (
        "run saved-actor walk-forward packaging before any promotion claim"
        if bool(selected["full_gate"])
        else "first-hit barrier labels did not solve the risk/underparticipation gap; next step should test sequence/state features or an execution-layer drawdown budget, not another BC model sweep"
    ),
}
(OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=True), encoding="utf-8")
selected_comparison[["window", "variant", "return", "relative_return", "max_dd", "avg_pos", "trades"]].sort_values(["window", "variant"])

,window,variant,return,relative_return,max_dd,avg_pos,trades
5,bad_C002_2025_rebound,barrier_bc,91.645517,-109.843614,16.324924,0.582175,10
4,bad_C002_2025_rebound,baseline,76.882481,-124.606649,16.324924,0.572851,9
7,bad_C003_2026_reentry,barrier_bc,17.799626,-217.582972,6.421473,0.161722,2
6,bad_C003_2026_reentry,baseline,17.799626,-217.582972,6.421473,0.161722,2
1,wf_2022,barrier_bc,-6.475865,80.112207,36.576424,0.350544,43
0,wf_2022,baseline,-21.567182,65.020890,31.096849,0.248277,26
3,wf_2023_2024,barrier_bc,307.364380,115.590871,24.260167,0.424908,96
2,wf_2023_2024,baseline,209.141657,17.368148,30.668184,0.497231,70


In [7]:
selected_audit

,window,train_rows,test_rows,barrier_positive,barrier_negative,derisk_pred,derisk_true,derisk_tp,derisk_fp,reentry_pred,reentry_true,reentry_tp,reentry_fp
0,wf_2022,484,251,57,188,0,10,0,0,55,53,20,35
1,wf_2023_2024,735,502,162,252,52,90,34,18,1,89,1,0
2,bad_C002_2025_rebound,1300,74,43,17,0,11,0,0,3,34,3,0
3,bad_C003_2026_reentry,1509,75,19,35,0,9,0,0,0,17,0,0


In [8]:
best = results.iloc[0]
pass_count = int(results["full_gate"].sum())
logistic_best = logistic_selection.get("selected_candidate", {})
tree_best = tree_selection.get("selected_candidate", {})
summary = {
    "candidate_count": int(len(results)),
    "full_gate_pass_count": pass_count,
    "best_mean_delta_relative_return": float(best["mean_delta_relative_return"]),
    "best_min_delta_relative_return": float(best["min_delta_relative_return"]),
    "best_max_key_window_dd": float(best["max_key_window_dd"]),
    "best_total_derisk_predictions": int(best["total_derisk_predictions"]),
    "best_total_reentry_predictions": int(best["total_reentry_predictions"]),
    "logistic_reference_mean_delta": logistic_best.get("mean_delta_return"),
    "logistic_reference_max_key_window_dd": logistic_best.get("max_key_window_dd"),
    "tree_reference_mean_delta": tree_best.get("mean_delta_relative_return"),
    "tree_reference_max_key_window_dd": tree_best.get("max_key_window_dd"),
}
conclusion = f"""## Runtime Conclusion

Decision: `{selection['decision']}`.

Screened `{summary['candidate_count']}` purged barrier-action BC candidates; full gate pass count `{summary['full_gate_pass_count']}`. Best candidate mean relative-return delta `{summary['best_mean_delta_relative_return']:.2f}pp`, min window delta `{summary['best_min_delta_relative_return']:.2f}pp`, and max key-window DD `{summary['best_max_key_window_dd']:.2f}%`.

Direct comparison: previous logistic label best mean delta `{summary['logistic_reference_mean_delta']:.2f}pp` with MaxDD `{summary['logistic_reference_max_key_window_dd']:.2f}%`; tree label best mean delta `{summary['tree_reference_mean_delta']:.2f}pp` with MaxDD `{summary['tree_reference_max_key_window_dd']:.2f}%`.

Mechanism readout: first-hit barrier labels test whether profit or stop loss happens first, which should reduce label ambiguity versus raw forward-return/drawdown labels. The selected path must still pass the same `+30pp` / `15%` notebook gate and no-worse-window guard. `selected_signal_audit.csv` is the overfit check: high false-positive action counts or sparse no-action candidates are not valid warm-start targets.

Keep/drop: keep `barrier_labels.py` as reusable label infrastructure. Do not promote this notebook replay directly. If the barrier label screen does not pass, the next useful iteration should add state/sequence features or an explicit execution-layer drawdown budget rather than another classifier family sweep.

Artifacts: `barrier_grid_results.csv`, `selected_comparison.csv`, `selected_signal_audit.csv`, `selected_paths.csv`, and `selection.json`.
"""
display(Markdown(conclusion))
print(json.dumps(summary, indent=2))

## Runtime Conclusion

Decision: `diagnostic_not_promoted`.

Screened `36` purged barrier-action BC candidates; full gate pass count `0`. Best candidate mean relative-return delta `32.02pp`, min window delta `0.00pp`, and max key-window DD `36.58%`.

Direct comparison: previous logistic label best mean delta `18.20pp` with MaxDD `30.67%`; tree label best mean delta `1.32pp` with MaxDD `31.10%`.

Mechanism readout: first-hit barrier labels test whether profit or stop loss happens first, which should reduce label ambiguity versus raw forward-return/drawdown labels. The selected path must still pass the same `+30pp` / `15%` notebook gate and no-worse-window guard. `selected_signal_audit.csv` is the overfit check: high false-positive action counts or sparse no-action candidates are not valid warm-start targets.

Keep/drop: keep `barrier_labels.py` as reusable label infrastructure. Do not promote this notebook replay directly. If the barrier label screen does not pass, the next useful iteration should add state/sequence features or an explicit execution-layer drawdown budget rather than another classifier family sweep.

Artifacts: `barrier_grid_results.csv`, `selected_comparison.csv`, `selected_signal_audit.csv`, `selected_paths.csv`, and `selection.json`.


{
  "candidate_count": 36,
  "full_gate_pass_count": 0,
  "best_mean_delta_relative_return": 32.01926899891025,
  "best_min_delta_relative_return": 0.0,
  "best_max_key_window_dd": 36.576424413602325,
  "best_total_derisk_predictions": 52,
  "best_total_reentry_predictions": 59,
  "logistic_reference_mean_delta": 18.197413391958435,
  "logistic_reference_max_key_window_dd": 30.66818414624621,
  "tree_reference_mean_delta": 1.3206534565801142,
  "tree_reference_max_key_window_dd": 31.096849210284773
}
